# Módulo 4: Clasificación - Predicción de Churn

## Contenido
1. El problema de clasificación
2. Preparación de datos
3. Importancia de features
4. Regresión logística
5. Entrenamiento con scikit-learn
6. Interpretación del modelo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, mutual_info_score
from sklearn.dummy import DummyClassifier

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Dataset de Churn

Simulamos datos de una empresa de telecomunicaciones.

In [ ]:
# Crear dataset sintético de churn
n = 2000
np.random.seed(42)

df = pd.DataFrame({
    'gender': np.random.choice(['Male', 'Female'], n),
    'tenure': np.random.randint(1, 72, n),
    'contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n, p=[0.5, 0.3, 0.2]),
    'monthly_charges': np.random.uniform(20, 100, n).round(2),
    'internet_service': np.random.choice(['DSL', 'Fiber optic', 'No'], n, p=[0.35, 0.45, 0.2]),
    'online_security': np.random.choice(['Yes', 'No', 'No internet'], n, p=[0.3, 0.5, 0.2]),
    'tech_support': np.random.choice(['Yes', 'No', 'No internet'], n, p=[0.3, 0.5, 0.2]),
    'payment_method': np.random.choice(
        ['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n
    ),
})

df['total_charges'] = (df['monthly_charges'] * df['tenure']).round(2)

# Generar churn basado en features (lógica realista)
prob_churn = (
    0.1
    + 0.3 * (df['contract'] == 'Month-to-month')
    - 0.15 * (df['contract'] == 'Two year')
    - 0.005 * df['tenure']
    + 0.003 * df['monthly_charges']
    + 0.1 * (df['internet_service'] == 'Fiber optic')
    - 0.1 * (df['online_security'] == 'Yes')
    + 0.05 * (df['payment_method'] == 'Electronic check')
)
prob_churn = prob_churn.clip(0.05, 0.95)
df['churn'] = (np.random.random(n) < prob_churn).astype(int)

print(f"Dataset: {df.shape}")
print(f"Tasa de churn: {df['churn'].mean():.2%}")
df.head()

## 2. Preparación de datos

In [ ]:
# Split
df_train_full, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_train_full, test_size=0.25, random_state=42)

y_train = df_train['churn'].values
y_val = df_val['churn'].values
y_test = df_test['churn'].values

# Quitar target de features
df_train = df_train.drop(columns=['churn'])
df_val = df_val.drop(columns=['churn'])
df_test = df_test.drop(columns=['churn'])

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")
print(f"Churn en train: {y_train.mean():.2%}")
print(f"Churn en val:   {y_val.mean():.2%}")

In [ ]:
# Identificar tipos
categoricas = df_train.select_dtypes(include='object').columns.tolist()
numericas = df_train.select_dtypes(include='number').columns.tolist()

print(f"Categóricas: {categoricas}")
print(f"Numéricas: {numericas}")

## 3. Importancia de Features

In [ ]:
# Risk Ratio por variable categórica
tasa_global = y_train.mean()
print(f"Tasa global de churn: {tasa_global:.2%}\n")

for col in categoricas:
    print(f"--- {col} ---")
    for val in df_train[col].unique():
        mask = df_train[col] == val
        tasa = y_train[mask.values].mean()
        rr = tasa / tasa_global
        print(f"  {val:<25}: tasa={tasa:.2%}, RR={rr:.2f}")
    print()

In [ ]:
# Mutual Information
mi_scores = {}
for col in categoricas:
    mi_scores[col] = mutual_info_score(df_train[col], y_train)

mi_df = pd.Series(mi_scores).sort_values(ascending=False)
print("Mutual Information:")
print(mi_df)

mi_df.plot(kind='barh', color='steelblue')
plt.xlabel('Mutual Information')
plt.title('Importancia de features categóricos')
plt.show()

In [ ]:
# Correlación de numéricas con churn
print("Correlación con churn:")
for col in numericas:
    corr = np.corrcoef(df_train[col].values, y_train)[0, 1]
    print(f"  {col:<20}: {corr:+.3f}")

## 4. Regresión Logística

In [ ]:
# La función sigmoide
def sigmoide(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-6, 6, 100)
plt.figure(figsize=(8, 4))
plt.plot(z, sigmoide(z), 'b-', lw=2)
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
plt.axvline(x=0, color='g', linestyle='--', alpha=0.5)
plt.xlabel('z (combinación lineal)')
plt.ylabel('P(y=1)')
plt.title('Función Sigmoide')
plt.grid(True, alpha=0.3)
plt.show()

## 5. Entrenamiento con scikit-learn

In [ ]:
# One-Hot Encoding con DictVectorizer
train_dicts = df_train[categoricas + numericas].to_dict(orient='records')
val_dicts = df_val[categoricas + numericas].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

print(f"Features después de encoding: {X_train.shape[1]}")
print(f"Algunos nombres: {dv.get_feature_names_out()[:10]}")

In [ ]:
# Entrenar modelo
modelo = LogisticRegression(solver='liblinear', max_iter=1000)
modelo.fit(X_train, y_train)

# Predecir
y_pred_proba = modelo.predict_proba(X_val)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

# Evaluar
acc = accuracy_score(y_val, y_pred)
auc = roc_auc_score(y_val, y_pred_proba)

print(f"Accuracy: {acc:.4f}")
print(f"AUC:      {auc:.4f}")

# Comparar con dummy
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
acc_dummy = accuracy_score(y_val, dummy.predict(X_val))
print(f"\nAccuracy dummy: {acc_dummy:.4f}")
print(f"Mejora sobre dummy: {(acc - acc_dummy)/acc_dummy*100:.1f}%")

## 6. Interpretación

In [ ]:
# Coeficientes del modelo
feature_names = dv.get_feature_names_out()
coefs = modelo.coef_[0]

importancia = pd.DataFrame({
    'feature': feature_names,
    'coef': coefs,
}).sort_values('coef', key=abs, ascending=False)

print("Top 15 features más importantes:")
print(importancia.head(15).to_string(index=False))

In [ ]:
# Visualizar
top = importancia.head(12)
colors = ['red' if c > 0 else 'blue' for c in top['coef']]

plt.figure(figsize=(10, 6))
plt.barh(top['feature'], top['coef'], color=colors)
plt.xlabel('Coeficiente')
plt.title('Importancia de Features (rojo=aumenta churn, azul=protege)')
plt.tight_layout()
plt.show()

In [ ]:
# Predecir para un cliente específico
cliente = {
    'contract': 'Month-to-month',
    'tenure': 3,
    'monthly_charges': 75.0,
    'total_charges': 225.0,
    'internet_service': 'Fiber optic',
    'online_security': 'No',
    'tech_support': 'No',
    'payment_method': 'Electronic check',
    'gender': 'Male',
}

X_cliente = dv.transform([cliente])
prob = modelo.predict_proba(X_cliente)[0, 1]
print(f"Probabilidad de churn: {prob:.1%}")
print(f"Decisión: {'CHURN' if prob >= 0.5 else 'NO CHURN'}")